In [ ]:
# Cell 1: Setup and File Inventory
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

project_root = Path().cwd().parent if Path().cwd().name == 'notebooks' else Path().cwd()
raw_path = project_root / 'data' / 'raw'

print("=" * 70)
print("DATA AUDIT - RAW FILE INVENTORY")
print("=" * 70)
print(f"Raw data path: {raw_path}\n")

for f in sorted(raw_path.glob('*')):
    if f.name.startswith('.') or f.name.startswith('~$'):
        continue
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f.name:<65} {size_mb:.2f} MB")

---
## Data Sources Overview

| # | File | Content | Period |
|---|------|---------|--------|
| 1 | Historical sales by store and by division | Weekly sales (42 stores, 6 categories) | FY2023-FY2025 |
| 2 | Budget_2023_.xlsx | Media spend budget | FY2023 |
| 3 | Budget 2024 - REEL au 5 novembre.xlsx | Media spend budget | FY2024 |
| 4 | Budget 2025 - 21 août.xlsx | Media spend budget | FY2025 |
| 5 | Recap_Tableau_Medias_2025.xlsx | Campaign performance metrics | FY2025 |
| 6 | CalendrierFiscal.xlsx | Fiscal calendar reference | FY2022-FY2027 |

**6 Product Categories (all equal):** HT, CR, SP, ME&GA, FI, BQ

**7 Media Channel Groups:** Television, Radio, Panneaux, Social_Media, Preroll, Banniere_Web, Circulaire_Digitale

In [ ]:
# Cell 2: AUDIT - Sales Data (Historical Sales)
print("=" * 70)
print("AUDIT 1: HISTORICAL SALES DATA")
print("=" * 70)

sales_file = raw_path / 'Historical sales by store and by division for 2023-2024-2025.xlsx'
df_sales = pd.read_excel(sales_file, sheet_name='Ventes cumulatives par magasin', header=1)

print(f"\nFile: {sales_file.name}")
print(f"Shape: {df_sales.shape[0]:,} rows x {df_sales.shape[1]} columns")
print(f"Stores: {df_sales['Code magasin'].nunique()} unique")
print(f"Date range: {df_sales['Commence le'].min().date()} to {df_sales['Commence le'].max().date()}")
print(f"Fiscal years: {sorted(df_sales['Année fiscale'].unique())}")

print(f"\nRows per fiscal year:")
for fy in sorted(df_sales['Année fiscale'].unique()):
    n = len(df_sales[df_sales['Année fiscale'] == fy])
    weeks = n // df_sales['Code magasin'].nunique()
    print(f"  FY{fy}: {n:,} rows ({weeks} weeks x 42 stores)")

print(f"\nColumns: {df_sales.columns.tolist()}")

print(f"\nMissing values:")
for col in df_sales.columns:
    n_missing = df_sales[col].isna().sum()
    if n_missing > 0:
        print(f"  {col}: {n_missing} ({n_missing/len(df_sales)*100:.1f}%)")

# Check for negative values (returns)
rev_cols = ['$-HT', '$-CR', '$-SP', '$-ME & $-GA', '$-FI', '$-BQ']
print(f"\nNegative values (returns):")
for col in rev_cols:
    n_neg = (df_sales[col] < 0).sum()
    if n_neg > 0:
        print(f"  {col}: {n_neg} rows ({n_neg/len(df_sales)*100:.1f}%)")

# Zero values
print(f"\nZero-value percentage:")
for col in rev_cols:
    n_zero = (df_sales[col] == 0).sum()
    pct = n_zero / len(df_sales) * 100
    print(f"  {col}: {pct:.1f}%")

print(f"\nSample stores: {sorted(df_sales['Code magasin'].unique())[:10]}")
print(f"\nDATA QUALITY: {'PASS' if df_sales.shape[0] == 6336 else 'CHECK'} - Expected 6,336 rows")

In [ ]:
# Cell 3: AUDIT - Budget Files (2023, 2024, 2025)
print("=" * 70)
print("AUDIT 2: BUDGET FILES")
print("=" * 70)

budget_files = [
    ('Budget_2023_.xlsx', 2023),
    ('Budget 2024 - REEL au 5 novembre.xlsx', 2024),
    ('Budget 2025 - 21 août.xlsx', 2025),
]

for fname, year in budget_files:
    fpath = raw_path / fname
    if not fpath.exists():
        print(f"\n  MISSING: {fname}")
        continue
    
    df = pd.read_excel(fpath, sheet_name=0, header=None)
    print(f"\n--- Budget {year}: {fname} ---")
    print(f"  Shape: {df.shape[0]} rows x {df.shape[1]} columns")
    
    # Show media names in column D (index 3)
    media_names = []
    for i in range(10, min(42, df.shape[0])):
        name = df.iloc[i, 3]
        if pd.notna(name) and str(name).strip():
            media_names.append(str(name).strip())
    print(f"  Media rows found: {len(media_names)}")
    print(f"  Media names: {media_names}")
    
    # Check month columns in row 6
    months_found = []
    for j in range(df.shape[1]):
        val = df.iloc[6, j] if 6 < df.shape[0] else None
        if pd.notna(val) and str(val).upper().strip() in [
            'NOVEMBRE', 'DECEMBRE', 'JANVIER', 'FEVRIER', 'MARS', 'AVRIL',
            'MAI', 'JUIN', 'JUILLET', 'AOUT', 'SEPTEMBRE', 'OCTOBRE'
        ]:
            months_found.append((j, str(val).strip()))
    print(f"  Month columns: {len(months_found)} found")
    
    # Check for duplicate month names (event sub-columns)
    month_names_only = [m[1].upper() for m in months_found]
    duplicates = [m for m in set(month_names_only) if month_names_only.count(m) > 1]
    if duplicates:
        print(f"  WARNING: Duplicate month columns: {duplicates}")
        print(f"  (Event sub-columns before monthly totals - code uses LAST match)")

print(f"\nDATA QUALITY: Audit complete for all budget files")

In [ ]:
# Cell 4: AUDIT - Tableau Medias & Calendrier Fiscal
print("=" * 70)
print("AUDIT 3: TABLEAU MEDIAS (Campaign Performance)")
print("=" * 70)

tm_file = raw_path / 'Recap_Tableau_Medias_2025.xlsx'
df_tm = pd.read_excel(tm_file, sheet_name='MASTER-TOTAL', header=0)
print(f"\nFile: {tm_file.name}")
print(f"Shape: {df_tm.shape}")
print(f"Columns: {df_tm.columns.tolist()[:10]}...")

# Check date range
date_col = df_tm.iloc[:, 1]
date_col = pd.to_datetime(date_col, errors='coerce')
print(f"Date range: {date_col.min()} to {date_col.max()}")

# Check for anomalous years
years = date_col.dt.year.dropna().unique()
print(f"Years in data: {sorted(years)}")
anomalous = [y for y in years if y < 2020 or y > 2026]
if anomalous:
    print(f"  WARNING: Anomalous year(s): {anomalous}")

# Media types
media_col = df_tm.iloc[:, 3]
print(f"\nMedia types: {media_col.dropna().unique().tolist()}")

# Total spend
cost_col = pd.to_numeric(df_tm.iloc[:, 11], errors='coerce')
print(f"Total spend: ${cost_col.sum():,.0f}")

print("\n" + "=" * 70)
print("AUDIT 4: CALENDRIER FISCAL")
print("=" * 70)

cal_file = raw_path / 'CalendrierFiscal.xlsx'
df_cal = pd.read_excel(cal_file, sheet_name='CalendrierFiscal', header=0)
print(f"\nFile: {cal_file.name}")
print(f"Shape: {df_cal.shape}")
print(f"Date range: {df_cal['Date'].min()} to {df_cal['Date'].max()}")
print(f"Fiscal years: {sorted(df_cal['Année fiscale'].dropna().unique())}")
print(f"Columns: {df_cal.columns.tolist()}")

print("\n" + "=" * 70)
print("AUDIT SUMMARY")
print("=" * 70)
print("""
All raw data files audited. Key findings:
- Sales: 6,336 weekly rows, 42 stores, FY2023-FY2025
- Budgets: 3 files covering FY2023-FY2025, 12 months each
- Tableau Medias: Campaign-level performance data
- Calendrier Fiscal: Daily fiscal calendar reference

Ready for cleaning in Notebook 02.
""")